In [1]:
%load_ext autoreload
%autoreload 2

**Author:** Salvador Navas  
**Date:** 2025-06-27

In [2]:
from pyhydra.data_sources.river_discharge import (
    download_usgs,
    get_usgs_site_info,
    search_usgs_sites,
)
import matplotlib.pyplot as plt
import pandas as pd

# 🇺🇸 USGS NWIS — National Water Information System

## 📚 General Description

The **USGS National Water Information System (NWIS)** provides free, real-time and historical streamflow data for thousands of gauging stations across the United States via a public REST API — **no registration required**.

---

## 📦 Variables available

| Parameter code | Variable | Unit |
|---------------|----------|------|
| `00060` | Discharge (default) | ft³/s → m³/s |
| `00065` | Gage height | ft |
| `00010` | Water temperature | °C |
| `00300` | Dissolved oxygen | mg/L |

## 📌 API endpoint

```
https://waterservices.usgs.gov/nwis/dv/
```

Documentation: https://waterservices.usgs.gov/docs/daily-values/

---
## 1. 🔍 Search for stations in a region

In [3]:
# Bounding box (west, south, east, north) in decimal degrees
BBOX = (-107.0, 35.5, -105.5, 37.5)   # Northern New Mexico

stations = search_usgs_sites(bbox=BBOX)
print(f"Stations found: {len(stations)}")
stations

Stations found: 160


---
## 2. ℹ️ Station metadata

In [4]:
SITE = "08279500"   # Rio Grande at Embudo, NM  — update as needed

info = get_usgs_site_info(SITE)
print(info.T)

                                       0
site_no                         08279500
station_nm      RIO GRANDE AT EMBUDO, NM
dec_lat_va                     36.205556
dec_long_va                  -105.963972
drain_area_va                      10400
drain_area_km2                 26935.896


---
## 3. ⬇️ Download discharge data

In [5]:
# === CONFIGURATION ===
SITE       = "08279500"
START_DATE = "1980-01-01"
END_DATE   = "2020-12-31"
UNITS      = "metric"       # 'metric' → m³/s  |  'imperial' → ft³/s

df = download_usgs(
    site_no=SITE,
    start_date=START_DATE,
    end_date=END_DATE,
    units=UNITS,
)

Q_col = f"Q_{SITE}"
print(f"Downloaded : {len(df)} days")
print(f"Missing    : {df[Q_col].isna().sum()} days  ({100*df[Q_col].isna().mean():.1f} %)")
print(f"Mean Q     : {df[Q_col].mean():.2f} m³/s")
print(f"Max  Q     : {df[Q_col].max():.1f} m³/s")
df.head()

Downloaded : 14976 days
Missing    : 0 days  (0.0 %)
Mean Q     : 22.71 m³/s
Max  Q     : 258.2 m³/s


In [6]:
# Download multiple sites at once
# df_multi = download_usgs(
#     site_no=["08279500", "08313000", "08317400"],
#     start_date="2000-01-01",
#     end_date="2020-12-31",
#     units="metric",
# )
# df_multi.describe()
print("(Uncomment to download multiple sites simultaneously)")

(Uncomment to download multiple sites simultaneously)


---
## 4. 📊 Visualisation

In [7]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Daily hydrograph
axes[0].plot(df.index, df[Q_col], lw=0.5, color="steelblue", alpha=0.8)
axes[0].set_ylabel("Daily discharge (m³/s)")
axes[0].set_title(f"USGS {SITE} — Rio Grande at Embudo, NM", fontsize=13)
axes[0].grid(alpha=0.3)

# Annual mean
annual = df[Q_col].resample("YE").mean()
axes[1].bar(annual.index.year, annual.values, color="steelblue", alpha=0.8)
axes[1].axhline(annual.mean(), color="red", lw=1.2, ls="--",
                label=f"Long-term mean: {annual.mean():.1f} m³/s")
axes[1].set_ylabel("Mean annual discharge (m³/s)")
axes[1].legend()
axes[1].grid(alpha=0.3)

# Seasonal regime
monthly = df[Q_col].groupby(df.index.month).mean()
axes[2].bar(monthly.index, monthly.values, color="steelblue", alpha=0.85)
axes[2].set_xticks(range(1, 13))
axes[2].set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun",
                          "Jul","Aug","Sep","Oct","Nov","Dec"])
axes[2].set_ylabel("Mean discharge (m³/s)")
axes[2].set_xlabel("Month")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [8]:
# Flow duration curve
import numpy as np

Q_sorted  = df[Q_col].dropna().sort_values(ascending=False)
exceedance = 100 * np.arange(1, len(Q_sorted) + 1) / len(Q_sorted)

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogy(exceedance, Q_sorted.values, color="steelblue", lw=1.5)
for p, label in [(10, "Q10"), (50, "Q50"), (90, "Q90")]:
    idx = int(p / 100 * len(Q_sorted))
    qval = Q_sorted.iloc[idx]
    ax.axvline(p, color="gray", ls="--", lw=0.8)
    ax.annotate(f"{label}={qval:.1f}", xy=(p+0.5, qval), fontsize=9)
ax.set_xlabel("Exceedance probability (%)")
ax.set_ylabel("Discharge (m³/s)")
ax.set_title(f"Flow duration curve · USGS {SITE}", fontsize=12)
ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()